This notebook processes ridership data from multiple transit agencies to create representative stop-level datasets for weekdays, Saturdays, and Sundays. For each agency, the workflow generally includes:

- Adding day type based on the service date (Weekday, Saturday, Sunday), with holidays normalized or removed as needed.
- Cleaning and renaming columns to ensure consistency across datasets (stop IDs, stop names, route names, and ridership fields).
- Summing boardings and alightings across directions at each stop, and aggregating multiple time periods where applicable.
- Calculating weighted averages when ridership is reported over multiple periods, using the number of service days as weights.
- Setting a ridership basis flag to indicate that values represent reported daily boardings and/or alightings.
- Computing final average ridership per stop, route, and day type, producing clean, representative stop-level datasets ready for modeling, analysis, or comparison across agencies.

This approach standardizes ridership data across agencies with differing data formats, missing values, and service coverage, enabling consistent cross-agency analysis.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")


In [5]:
def add_day_type(df, date_col, output_col='day_type'):
    """
    Classify each row as 'Weekday', 'Saturday', or 'Sunday' based on a specified date column.
    This is useful when daily datasets include labels such as 'Holiday' or when no day_type
    column exists. The function converts the date column, identifies the day of week, and
    standardizes the day-type classification accordingly. Either a start_date or end_date
    column can be used for datasets with daily reporting.
    """
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [6]:
# Counting number of days without holidays
def count_service_days(row):
    
    if str(row.day_type).lower() == "all":
        return (row.end_date - row.start_date).days + 1

    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum(d.weekday() < 5 for d in dates)

    elif row.day_type == "Saturday":
        return sum(d.weekday() == 5 for d in dates)

    elif row.day_type == "Sunday":
        return sum(d.weekday() == 6 for d in dates)

In [7]:
# Counting number of days with holidays
def count_service_days_wo_holidays(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum((d.weekday() < 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Saturday":
        return sum((d.weekday() == 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Sunday":
        return sum((d.weekday() == 6) and (d not in holiday_set) for d in dates)

In [8]:
def compute_averages(df, ridership_cols, group_cols):
    """
    Computes average of one or more ridership columns per group in long format.

    Parameters:
    - df: pandas DataFrame
    - ridership_cols: list of columns to average (e.g., ['daily_boardings', 'daily_alightings'])
    - group_cols: list of columns to group by (e.g., ['stop_name', 'day_type'])

    Returns:
    - DataFrame with columns: group_cols + average of each ridership column
    """
    
    if isinstance(ridership_cols, str):
        ridership_cols = [ridership_cols]  # allow single string input
    
    averaged = df.groupby(group_cols)[ridership_cols].mean().reset_index()
    
    # rename each column to indicate it's averaged
    averaged = averaged.rename(columns={col: f"average_{col}" for col in ridership_cols})
    
    return averaged

In [9]:
master_cols = [
    "organization_name",
    "route_id",
    "route_name",
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "day_type",
    "average_daily_boardings",
    "average_daily_alightings"
]

def standardize_columns(df, master_cols):
    
    # add missing columns
    for col in master_cols:
        if col not in df.columns:
            df[col] = None

    # keep only master columns and order them
    return df[master_cols]

In [10]:
bart_ridership = add_day_type(bart_ridership, date_col='start_date')
bart_ridership['daily_boardings'] = pd.to_numeric(bart_ridership['daily_boardings'], errors='coerce')
bart_ridership['daily_alightings'] = pd.to_numeric(bart_ridership['daily_alightings'], errors='coerce')
average_bart_ridership = compute_averages(
    df=bart_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["dataset_id", "organization_name", 
                "stop_id", "stop_name", "stop_lat", "stop_lon", "day_type"])
    
bart_ridership_final = average_bart_ridership[['organization_name', 'stop_id', 
                                               'stop_name', 'day_type', 'average_daily_boardings',
                                              'average_daily_alightings']]

bart_ridership_final = standardize_columns(bart_ridership_final, master_cols)

bart_ridership_final.sample(5)

/tmp/ipykernel_4206/1558764047.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = None
/tmp/ipykernel_4206/1558764047.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = None


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
15,San Francisco Bay Area Rapid Transit District,None,None,901209,Montgomery Street,None,None,Saturday,4387.942308,5476.269231
114,San Francisco Bay Area Rapid Transit District,None,None,905309,Dublin / Pleasanton,None,None,Saturday,1829.673077,1850.980769
19,San Francisco Bay Area Rapid Transit District,None,None,901309,Powell Street,None,None,Sunday,7946.634615,6790.500000
53,San Francisco Bay Area Rapid Transit District,None,None,902509,Bayfair,None,None,Weekday,2554.141762,2598.915709
129,San Francisco Bay Area Rapid Transit District,None,None,907109,San Francisco International Airport (SFO),None,None,Saturday,3801.250000,2974.961538


In [11]:
bart_ridership_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
62,San Francisco Bay Area Rapid Transit District,None,None,902809,Union City,None,None,Weekday,1842.532567,1855.567050
124,San Francisco Bay Area Rapid Transit District,None,None,906309,San Bruno,None,None,Sunday,614.076923,681.961538
104,San Francisco Bay Area Rapid Transit District,None,None,904509,El Cerrito Del Norte,None,None,Weekday,3592.773946,3824.873563
76,San Francisco Bay Area Rapid Transit District,None,None,903409,Walnut Creek,None,None,Sunday,1213.846154,1233.173077
68,San Francisco Bay Area Rapid Transit District,None,None,903109,Rockridge,None,None,Weekday,2623.076628,2671.118774


In [12]:
big_blue_bus_ridership.columns = big_blue_bus_ridership.columns.str.lower()
big_blue_bus_ridership['service_day'] = big_blue_bus_ridership['service_day'].str.strip().str.capitalize()


big_blue_bus_ridership = big_blue_bus_ridership.rename(columns={
    'route_number': 'route_id',
    'service_day': 'day_type',
    'average_daily_boardings':'average_daily_boardings_period',
    'average_daily_alightings':'average_daily_alightings_period'    
})


# Sum boardings/alightings across directions within same stop & period
total_big_blue_bus_ridership = (
    big_blue_bus_ridership
    .groupby(['day_type','route_id','route_name',
              'stop_id','stop_name','stop_lat','stop_lon',
              'start_date','end_date'])
    .agg({
        'average_daily_boardings_period':'sum',
        'average_daily_alightings_period':'sum'
    })
    .reset_index()
)


In [13]:
big_blue_bus_ridership['start_date'] = pd.to_datetime(big_blue_bus_ridership['start_date'])
big_blue_bus_ridership['end_date'] = pd.to_datetime(big_blue_bus_ridership['end_date'])

total_big_blue_bus_ridership["n_days"] = total_big_blue_bus_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 4 different time periods
averaged_total_big_blue_bus_ridership = (
    total_big_blue_bus_ridership
    .groupby(['day_type','route_id','route_name', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['average_daily_alightings_period'], weights=x['n_days'])
    }))
    .reset_index()
)


# Set the ridership basis flag
averaged_total_big_blue_bus_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_total_big_blue_bus_ridership["organization_name"] = "Big Blue Bus"

big_blue_bus_ridership_final = standardize_columns(averaged_total_big_blue_bus_ridership, master_cols)
big_blue_bus_ridership_final.sample(5)

/tmp/ipykernel_4206/2506038505.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
750,Big Blue Bus,18,UCLA - Abbot Kinney,2931,ROSE WB/RUTH NS,33.999649,-118.471270,Saturday,5.051564,1.807754
2842,Big Blue Bus,27,Pico Blvd Rapid,1343,PICO WB/18TH NS,34.018748,-118.471092,Weekday,18.672356,53.372618
953,Big Blue Bus,2,Wilshire Blvd/UCLA,2581,WESTWOOD PLAZA/STEIN PLAZA DRIVEWAY,34.065305,-118.445200,Sunday,2.706650,26.453554
1878,Big Blue Bus,3,Lincoln Blvd/LAX,2608,LINCOLN NB/MANCHESTER FS,33.960504,-118.419710,Weekday,56.497412,52.927517
2848,Big Blue Bus,27,Pico Blvd Rapid,2331,PICO WB/LA BREA NS,34.048021,-118.344337,Weekday,22.799838,7.082820


In [14]:
caltrain_ridership = caltrain_ridership.rename(columns={
    'Origin Station': 'stop_name',
    'Date Type': 'day_type',
    'Average Ridership':'average_daily_boardings_period',  
})

caltrain_ridership = caltrain_ridership[
    caltrain_ridership["day_type"] != "Holiday"
]

# Getting number of days and using US Federal Holidays
start_date_caltrain = '2023-11-01'
end_date_caltrain = '2025-07-31'

#Getting all the federal holidays between the start and end date
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=start_date_caltrain, end=end_date_caltrain)

# Setting holidays to calculate the number of weekdays, saturday and sunday without holidays 
holiday_set = set(holidays)

caltrain_ridership["start_date"] = pd.to_datetime(caltrain_ridership["start_date"])
caltrain_ridership["end_date"] = pd.to_datetime(caltrain_ridership["end_date"])

caltrain_ridership["n_days"] = caltrain_ridership.apply(count_service_days_wo_holidays, axis=1)

In [15]:
# Taking weighted average of the ridership across 20 months time periods
averaged_caltrain_ridership = (
    caltrain_ridership
    .groupby(['day_type','stop_name'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days'])
    }))
    .reset_index()
)

# Set the ridership basis flag
averaged_caltrain_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_caltrain_ridership["organization_name"] = "Caltrain"

caltrain_ridership_final = standardize_columns(averaged_caltrain_ridership, master_cols)

caltrain_ridership_final.sample(5)

/tmp/ipykernel_4206/3512310577.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
24,Caltrain,None,None,None,San Martin,None,None,Saturday,0.084733,None
70,Caltrain,None,None,None,Hayward Park,None,None,Weekday,291.195338,None
78,Caltrain,None,None,None,Redwood City,None,None,Weekday,1802.931211,None
2,Caltrain,None,None,None,Belmont,None,None,Saturday,252.710547,None
27,Caltrain,None,None,None,South San Francisco,None,None,Saturday,203.253125,None


In [16]:
culver_citybus_ridership["daily_ridership_basis"] = "reported_avg_daily"
culver_citybus_ridership["Route"] = culver_citybus_ridership["Route"].str.split("-", n=1).str[1]


culver_citybus_ridership = culver_citybus_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'route_id': 'route_id',
    'Route': 'route_name',
})

group_cols = [
    "route_name", "stop_id", "stop_name", "route_id", "day_type", "daily_ridership_basis"]

# Sum AVG On and AVG Off
culver_citybus_ridership = (culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("AVG On", "sum"),
        average_daily_alightings=("AVG Off", "sum")
    )
)

culver_citybus_ridership["organization_name"] = "Culver City Bus"

culver_citybus_final = standardize_columns(culver_citybus_ridership, master_cols)

culver_citybus_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
676,Culver City Bus,6,Sepulveda Boulevard,656,SepulvedaBlvd/ExpositionBlvd,None,None,Weekday,252.7,145.4
949,Culver City Bus,1,Washington Boulevard,154,WashingtonBlvd/OverlandAve,None,None,Weekday,72.8,49.5
175,Culver City Bus,3,Crosstown,369,Overland Ave/Braddock Dr,None,None,Weekday,6.0,7.0
992,Culver City Bus,1,Washington Boulevard,169,Washington Blvd/Redwood Ave,None,None,Saturday,10.9,25.7
721,Culver City Bus,6,Sepulveda Boulevard,669,Sepulveda Blvd/Studio Village West,None,None,Sunday,5.7,11.6


In [17]:
# Add day_type from the date column
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col="date")

foothill_transit_ridership = foothill_transit_ridership.rename(columns={
    'route_short_name': 'route_id',
    'stop_code': 'stop_id',
})

# Grouping columns
group_cols = [
    "date", "route_id", "stop_id", "stop_lat", "stop_lon",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_ridership_foothill = (
    foothill_transit_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("boardings", "sum"),
        daily_alightings=("alightings", "sum")
    )
)

# Set the ridership basis flag
total_ridership_foothill["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_ridership_foothill['daily_boardings'] = pd.to_numeric(total_ridership_foothill['daily_boardings'], errors='coerce')
total_ridership_foothill['daily_alightings'] = pd.to_numeric(total_ridership_foothill['daily_alightings'], errors='coerce')

average_foothill_ridership = compute_averages(
    df=total_ridership_foothill,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_id", "stop_id", 
                "stop_lat", "stop_lon", "day_type"]
)

average_foothill_ridership["organization_name"] = "Foothill Transit"
foothill_final = standardize_columns(average_foothill_ridership, master_cols)

foothill_final.sample(5)


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
363,Foothill Transit,178,None,2963,None,34.040200,-117.926566,Weekday,3.921875,0.738281
485,Foothill Transit,185,None,1658,None,34.037477,-117.950057,Weekday,28.321839,29.743295
5862,Foothill Transit,488,None,3133,None,34.072665,-118.032247,Weekday,16.176245,1.164751
4701,Foothill Transit,480,None,977,None,34.094797,-117.705128,Saturday,1.333333,1.000000
2296,Foothill Transit,269,None,2395,None,34.044159,-118.046234,Sunday,1.526316,0.263158


In [18]:
# Add day_type from the date column
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col="Date")

fresno_area_express_ridership["daily_ridership_basis"] = "reported_daily"


fresno_area_express_ridership = fresno_area_express_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopLabel': 'stop_name',
    'ProjectedBoarding': 'daily_boardings',
    'ProjectedAlighting': 'daily_alightings'
})

# Calculate average ridership per day type and stop
fresno_area_express_ridership['daily_boardings'] = pd.to_numeric(fresno_area_express_ridership['daily_boardings'], errors='coerce')
fresno_area_express_ridership['daily_alightings'] = pd.to_numeric(fresno_area_express_ridership['daily_alightings'], errors='coerce')

average_fresno_area_express_ridership = compute_averages(
    df=fresno_area_express_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id",  "stop_name", "day_type"]
)

average_fresno_area_express_ridership["organization_name"] = "Fresno County"

fresno_final = standardize_columns(average_fresno_area_express_ridership, master_cols)

fresno_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
4706,Fresno County,None,None,2453,SE Olive - Highway 41,None,None,Saturday,1.412294,3.055589
3360,Fresno County,None,None,1575,SW MARKS - GETTYSBURG,None,None,Saturday,7.425946,2.512621
1502,Fresno County,None,None,734,SW PALM - OLIVE,None,None,Sunday,10.496907,16.431031
320,Fresno County,None,None,168,NW CEDAR - SHIELDS,None,None,Sunday,38.163266,40.540738
1905,Fresno County,None,None,900,SW WISHON - FERN,None,None,Weekday,22.253657,51.948229


In [19]:
gold_coast_transit_ridership = gold_coast_transit_ridership.rename(columns={
    'day_of_week': 'day_type',
    'route': 'route_id',
    'lat': 'stop_lat',
    'lon': 'stop_lon'
})

group_cols = ["day_type", "route_id", "stop_id", "stop_name",  "stop_lat", "stop_lon", "start_date", "end_date"]

# Sum AVG On 
total_gold_coast_transit_ridership = (gold_coast_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("total_on", "sum"),
        daily_alightings=("total_off", "sum"),
    )
)

total_gold_coast_transit_ridership['start_date'] = pd.to_datetime(total_gold_coast_transit_ridership['start_date'])
total_gold_coast_transit_ridership['end_date'] = pd.to_datetime(total_gold_coast_transit_ridership['end_date'])

total_gold_coast_transit_ridership["n_days"] = total_gold_coast_transit_ridership.apply(count_service_days, axis=1)

In [20]:
# Taking weighted average of the ridership across 4 different time periods
averaged_gold_coast_transit_ridership = (
    total_gold_coast_transit_ridership
    .groupby(['day_type','route_id', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['daily_boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['daily_alightings'], weights=x['n_days'])
    }))
    .reset_index()
)

averaged_gold_coast_transit_ridership["organization_name"] = "Gold Coast Transit"
golden_coast_transit_final = standardize_columns(averaged_gold_coast_transit_ridership, master_cols)

golden_coast_transit_final.sample(5)

/tmp/ipykernel_4206/2301941411.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
756,Gold Coast Transit,18A,None,CIBLAS1,Channel Isl & Lassen,34.175577,-119.190003,Weekday,1.0,0.0
326,Gold Coast Transit,10,None,TELVIC3,Telegraph & Victoria,34.278000,-119.213340,Weekday,28.0,11.0
559,Gold Coast Transit,17,None,ROSNCD1,Rose & North Campus,34.166521,-119.159594,Weekday,5.0,18.0
335,Gold Coast Transit,10,None,VIOLLA2,Violeta & LA Ave,34.284324,-119.148805,Weekday,7.0,9.0
294,Gold Coast Transit,10,None,TELBMS1,Telegraph & Bryn Mawr,34.276926,-119.222292,Weekday,2.0,3.0


In [21]:
# Add day_type from the date column
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col="Date")

group_cols = [
    "Date", "day_type", "Stop", "start_date", "end_date"]

# Sum AVG On 
total_golden_gate_park_shuttle_ridership = (golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ridership", "sum"),
    )
)

total_golden_gate_park_shuttle_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_park_shuttle_ridership = total_golden_gate_park_shuttle_ridership.rename(columns={
    'Stop': 'stop_name',
    'Date': 'date'
})

# Calculate average ridership per day type and stop
total_golden_gate_park_shuttle_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_park_shuttle_ridership['daily_boardings'], errors='coerce')
average_golden_gate_park_shuttle_ridership = compute_averages(
    df=total_golden_gate_park_shuttle_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_name", "day_type"]
)


average_golden_gate_park_shuttle_ridership["organization_name"] = "Golden Gate Park Shuttle"
golden_gate_parkshuttle_final = standardize_columns(average_golden_gate_park_shuttle_ridership, master_cols)

golden_gate_parkshuttle_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
20,Golden Gate Park Shuttle,None,None,None,Conservatory of Flowers EB,None,None,Weekday,6.616858,None
12,Golden Gate Park Shuttle,None,None,None,Academy of Sciences,None,None,Saturday,13.288462,None
48,Golden Gate Park Shuttle,None,None,None,Tennis Center/ Dalia Dell WB,None,None,Saturday,18.615385,None
32,Golden Gate Park Shuttle,None,None,None,JFK Gateway EB,None,None,Weekday,3.551724,None
23,Golden Gate Park Shuttle,None,None,None,Conservatory of Flowers WB,None,None,Weekday,14.942529,None


In [22]:
# Add day_type from the date column
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, date_col="date")

golden_gate_transit_ridership = golden_gate_transit_ridership.rename(columns={
    'STOP_NUMBER': 'stop_id',
    'STOP_NAME': 'stop_name',
    'ROUTE': 'route_id',
})

# Remove everything in parentheses (including the parentheses) at the end of the string
golden_gate_transit_ridership['stop_name'] = golden_gate_transit_ridership['stop_name'].str.replace(r'\s*\(.*\)$', '', regex=True)

group_cols = [
    "day_type", "route_id", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("BOARDINGS", "sum"),
        daily_alightings=("ALIGHTINGS", "sum")
    )
)

total_golden_gate_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_golden_gate_transit_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_boardings'], errors='coerce')
total_golden_gate_transit_ridership['daily_alightings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_alightings'], errors='coerce')

average_golden_gate_transit_ridership = compute_averages(
    df=total_golden_gate_transit_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_id", "stop_id", "stop_name", "day_type"]
)

average_golden_gate_transit_ridership["organization_name"] = "Golden Gate Transit"

golden_gate_transit_final = standardize_columns(average_golden_gate_transit_ridership, master_cols)

golden_gate_transit_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
702,Golden Gate Transit,130,None,80036,VTP 101 NB @ Tiburon Ridge,None,None,Sunday,0.000000,0.0
254,Golden Gate Transit,101,None,44002,San Rafael Transit Center-Platform A,None,None,Sunday,213.000000,120.0
982,Golden Gate Transit,150,None,80044,VTP Lombard WB @ Van Ness,None,None,Weekday,0.000000,0.0
320,Golden Gate Transit,101,None,80039,VTP 101 SB @ Marin City,None,None,Sunday,0.000000,0.0
412,Golden Gate Transit,130,None,40003,Salesforce Transit Center-Bus Plaza Bay A,None,None,Weekday,37.681818,0.0


In [23]:
long_beach_transit_ridership["daily_ridership_basis"] = "reported_avg_daily"

group_cols = [
    "DayType", "Route", "StopID", "StopName", "end_date", "start_date", "daily_ridership_basis"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Boardings", "sum"),
        average_daily_alightings=("Alightings", "sum")
    )
)

total_long_beach_transit_ridership = total_long_beach_transit_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopName': 'stop_name',
    'DayType': 'day_type',
    'Route': 'route_id',
})

total_long_beach_transit_ridership["organization_name"] = "Long Beach Transit"

long_beach_transit_final = standardize_columns(total_long_beach_transit_ridership, master_cols)

long_beach_transit_final.sample(5)


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
213,Long Beach Transit,21,None,213,Cherry & Denmead NE,None,None,Saturday,1.597657,3.193276
6984,Long Beach Transit,92,None,4146,Woodruff & Alondra SW,None,None,Weekday,46.521502,0.000000
3364,Long Beach Transit,41,None,467,Anaheim & Martin Luther King Jr SE,None,None,Sunday,26.595215,9.946616
5966,Long Beach Transit,22,None,275,Ocean & Cherry NW,None,None,Weekday,8.802974,37.698834
2794,Long Beach Transit,192,None,570,Long Beach Blvd & Del Amo NE,None,None,Saturday,44.393854,28.840167


In [24]:
octa_ridership = octa_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Route': 'route_name'

})

octa_ridership["stop_name"] = octa_ridership["stop_name"].str.split("-", n=1).str[1]
octa_ridership["route_name"] = octa_ridership["route_name"].str.split("-", n=1).str[1]

group_cols = [
    "day_type", "route_name", "stop_id", "route_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
octa_ridership = (octa_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("APC Boarding", "sum"),
        average_daily_alightings=("APC Alighting", "sum")
    )
)

octa_ridership["organization_name"] = "Orange County Transportation Authority"

octa_final = standardize_columns(octa_ridership, master_cols)

octa_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
373,Orange County Transportation Authority,123,Anaheim to Huntington Beach,1468,VALLEY VIEW-CERRITOS,None,None,weekday,3,5
4850,Orange County Transportation Authority,70,Sunset Beach - Tustin,2024,EDINGER-WARD,None,None,weekday,3,3
749,Orange County Transportation Authority,26,Buena Park - Yorba Linda,5756,PLACENTIA-CHAPMAN,None,None,weekday,13,12
3529,Orange County Transportation Authority,46,Long Beach - Orange,5583,TAFT-O'DONNELL,None,None,weekday,0,5
3792,Orange County Transportation Authority,1,Long Beach - San Clemente,6856,PACIFIC COAST-12TH,None,None,weekday,2,1


In [25]:
omni_trans_ridership= omni_trans_ridership.rename(columns={
    'Route': 'route_id',
    'Stop Name': 'stop_name',
    'avg_boardings': 'average_daily_boardings',
    'avg_alightings': 'average_daily_alightings'
})

omni_trans_ridership["organization_name"] = "Omnitrans"

omni_trans_ridership = omni_trans_ridership[['route_id', 'stop_name', 'average_daily_boardings', 'average_daily_alightings', 'organization_name']]

omni_final = standardize_columns(omni_trans_ridership, master_cols)

omni_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
4005,Omnitrans,19,None,None,M @ Fogg,None,None,None,0.046575,0.063014
863,Omnitrans,22,None,None,Riverside @ Gateway,None,None,None,0.465753,3.487671
1686,Omnitrans,1,None,None,Sierra Way @ Court,None,None,None,2.616438,2.536986
563,Omnitrans,14,None,None,San Bernardino Transit Center,None,None,None,189.884932,173.887671
2306,Omnitrans,15,None,None,San Bernardino @ Alabama,None,None,None,3.046575,1.758904


In [26]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    "date", "Stop ID", "Route", "start_date", 
    "end_date", "day_type"]

# Sum AVG On and AVG Off
total_riverside_transit_ridership = (riverside_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    daily_boardings = ('ridership', 'sum'))
)

total_riverside_transit_ridership = total_riverside_transit_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_id'

})

In [27]:
total_riverside_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_riverside_transit_ridership['daily_boardings'] = pd.to_numeric(total_riverside_transit_ridership['daily_boardings'], errors='coerce')
average_riverside_transit_ridership = compute_averages(
    df=total_riverside_transit_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["route_id", "stop_id", "day_type"]
)

average_riverside_transit_ridership["organization_name"] = "Riverside Transit"

riverside_final = standardize_columns(average_riverside_transit_ridership, master_cols)

riverside_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
6791,Riverside Transit,41,None,3186,None,None,None,Weekday,2.288889,None
224,Riverside Transit,1,None,1126,None,None,None,Weekday,13.277174,None
5127,Riverside Transit,23,None,1286,None,None,None,Sunday,3.558824,None
2759,Riverside Transit,13,None,1715,None,None,None,Saturday,2.750000,None
3789,Riverside Transit,18,None,2030,None,None,None,Weekday,2.447552,None


In [30]:
sacrt_bus_ridership = sacrt_bus_ridership.rename(columns={
    'route_long_name': 'route_name',
})

group_cols = [
    "stop_id", "stop_lat", "stop_lon", "stop_name", "route_name", "route_id", "day_type"]

# Combining across directions
sacrt_bus_ridership = (sacrt_bus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("average_boardings", "sum"),
        average_daily_alightings=("average_alightings", "sum")
    )
)

sacrt_bus_ridership["organization_name"] = "SacRT Bus"

sacrt_bus_final = standardize_columns(sacrt_bus_ridership, master_cols)

sacrt_bus_final.sample(5)


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
3347,SacRT Bus,067,FRANKLIN,4015,WYNDHAM DR & BRUCEVILLE RD (WB),38.469135,-121.419777,daily,2,1
3731,SacRT Bus,215,14TH AVE - FLORIN TOWNE,5330,VANDENBERG DR & 79TH ST (EB),38.527870,-121.412252,weekday,0,0
1392,SacRT Bus,129,ARDEN COMMUTER,1292,CALIFORNIA AVE & FAIR OAKS BLVD (NB),38.637336,-121.320965,weekday,0,0
3521,SacRT Bus,248,MEADOWVIEW RD - RUSH RIVER DR,4364,GLORIA DR & RIVERGATE WAY (NB),38.496873,-121.543634,weekday,0,0
2411,SacRT Bus,206,12TH AVE - SUTTERVILLE RD,2506,12TH AVE & FRANKLIN BLVD (EB),38.541052,-121.475363,weekday,0,0


In [32]:
samtrans_ridership = add_day_type(samtrans_ridership, date_col='APC Date')

samtrans_ridership = samtrans_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Route': 'route_id',
    'Lat': 'stop_lat',
    'Lon': 'stop_lon',
    'APC Date': 'date',
    'Ons': 'daily_boardings',
    'Offs': 'daily_alightings',
})

samtrans_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
samtrans_ridership['daily_boardings'] = pd.to_numeric(samtrans_ridership['daily_boardings'], errors='coerce')
samtrans_ridership['daily_alightings'] = pd.to_numeric(samtrans_ridership['daily_alightings'], errors='coerce')

average_samtrans_ridership = compute_averages(
    df=samtrans_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_id", "stop_id", "stop_name", "stop_lat", "stop_lon", "day_type"]
)

average_samtrans_ridership["organization_name"] = "Samtrans"

samtrans_final = standardize_columns(average_samtrans_ridership, master_cols)

samtrans_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
1730,Samtrans,130,None,334639,Gateway Blvd & Larkspur Landing,37.659595,-122.396934,Saturday,5.2,0.0
3038,Samtrans,281,None,347064,University Ave & Middlefield Rd,37.450885,-122.156600,Sunday,4.0,2.2
3887,Samtrans,296,None,344211,3500 Middlefield Rd-St Anthonys Chu,37.470344,-122.199501,Sunday,10.0,16.0
6500,Samtrans,ECRO,None,335160,San Bruno Ave W & Green Ave,37.629501,-122.414642,Sunday,1.5,1.0
4723,Samtrans,397,None,344191,Middlefield Rd & 2nd Ave,37.474216,-122.205909,Saturday,1.0,0.0


In [33]:
group_cols = [
    "Route", "Route Name", "Stop ID",  "Stop Name", "day_type", "start_date", 
    "end_date"]

# Sum AVG On and AVG Off
total_sdmts_ridership = (sdmts_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Average On", "sum"),
        average_daily_alightings=("Average Off", "sum")
    )
)

total_sdmts_ridership["Route Name"] = total_sdmts_ridership["Route Name"].str.split(":", n=1).str[1]

total_sdmts_ridership = total_sdmts_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_id',
    'Route Name': 'route_name',
    'Stop Name': 'stop_name',
})

total_sdmts_ridership["daily_ridership_basis"] = "reported_avg_daily"

total_sdmts_ridership["organization_name"] = "SDMTS"

sdmts_final = standardize_columns(total_sdmts_ridership, master_cols)

sdmts_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
11920,SDMTS,933,Iris TC Loop-Imperial Beach Counterclock,89028,Iris Avenue Transit Center,None,None,Sunday,0.000000,161.171703
6505,SDMTS,520,NaN,75038,Lemon Grove Depot,None,None,Saturday,917.044107,1702.601671
11405,SDMTS,929,Downtown-Iris Transit Center,50037,30th St & Highland Av,None,None,Saturday,2.000000,2.000000
2246,SDMTS,11,SDSU-Downtown San Diego,11640,2nd Av & Beech St,None,None,Saturday,6.281555,28.658438
13201,SDMTS,961,24th St Transit Center-Encanto Trolley,99493,Sweetwater Rd & Town & Country,None,None,Weekday,18.781038,53.785627


In [34]:
santa_cruz_metro_ridership = santa_cruz_metro_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Boardings': 'average_daily_boardings',
    'Alightings': 'average_daily_alightings'    
})

santa_cruz_metro_ridership["organization_name"] = "Santa Cruz Metro"

santa_cruz_final = standardize_columns(santa_cruz_metro_ridership, master_cols)

santa_cruz_final.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
511,Santa Cruz Metro,None,None,1651,Mt Hermon Rd + Lockhart Gulch Rd,None,None,all,0.263014,0.452055
239,Santa Cruz Metro,None,None,2310,Empire Grade (#2519),None,None,all,0.136986,0.005479
507,Santa Cruz Metro,None,None,1648,Mt Hermon Rd + Conference Dr,None,None,all,0.183562,0.150685
215,Santa Cruz Metro,None,None,1360,E Cliff Dr + 13th Ave,None,None,all,1.821918,0.673973
116,Santa Cruz Metro,None,None,2271,Broadway + S Branciforte Ave,None,None,all,2.282192,3.772603


In [35]:
sbmtd_ridership = sbmtd_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'avg_boardings': 'daily_boardings',
    'avg_ridership': 'daily_alightings'    
})


sbmtd_ridership["n_days"] = sbmtd_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 12 months
averaged_sbmtd_ridership = (
    sbmtd_ridership
    .groupby(['stop_id', 'stop_name', 'day_type'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['Ridership by Stop_Boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['Ridership by Stop_Alightings'], weights=x['n_days'])
    }))
    .reset_index()
)



averaged_sbmtd_ridership["organization_name"] = "SBMTD"

sbmtd_final = standardize_columns(averaged_sbmtd_ridership, master_cols)

sbmtd_final.sample(5)

/tmp/ipykernel_4206/3484305919.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
264,SBMTD,None,None,311,Carpinteria & Maple,None,None,all,86.528767,536.441096
244,SBMTD,None,None,291,Haley & Laguna,None,None,all,69.265753,105.197260
152,SBMTD,None,None,192,Milpas & Quinientos,None,None,all,258.276712,302.167123
44,SBMTD,None,None,58,State & Mission,None,None,all,548.019178,1005.684932
65,SBMTD,None,None,81,San Andres & Valerio,None,None,all,580.586301,171.498630


In [36]:
all_ridership = pd.concat([
    bart_ridership_final,
    big_blue_bus_ridership_final,
    caltrain_ridership_final,
    culver_citybus_final,
    foothill_final,
    fresno_final,
    golden_coast_transit_final,
    golden_gate_parkshuttle_final,
    golden_gate_transit_final,
    long_beach_transit_final,
    octa_final,
    omni_final,
    riverside_final,
    sacrt_bus_final,
    samtrans_final,
    sdmts_final,
    santa_cruz_final,
    sbmtd_final,
], ignore_index=True)

/tmp/ipykernel_4206/4200524122.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_ridership = pd.concat([


In [37]:
all_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71703 entries, 0 to 71702
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         71703 non-null  object 
 1   route_id                  65173 non-null  object 
 2   route_name                27000 non-null  object 
 3   stop_id                   66727 non-null  object 
 4   stop_name                 56894 non-null  object 
 5   stop_lat                  21395 non-null  float64
 6   stop_lon                  21395 non-null  float64
 7   day_type                  66871 non-null  object 
 8   average_daily_boardings   71703 non-null  float64
 9   average_daily_alightings  63461 non-null  float64
dtypes: float64(4), object(6)
memory usage: 5.5+ MB


In [38]:
all_ridership.sample(5)

,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
20943,Long Beach Transit,182,None,4012,1st & Shelter H S,NaN,NaN,Saturday,0.000000,82.912442
63621,SDMTS,704,E St TC-Palomar TC,30323,Medical Center Dr & E Naples St,NaN,NaN,Weekday,3.847581,4.930707
6375,Foothill Transit,195,None,2985,None,34.033068,-117.756849,Sunday,2.333333,0.769231
65284,SDMTS,855,Rancho San Diego-La Mesa,40286,Campo Rd & Conrad Dr,NaN,NaN,Weekday,54.865839,38.105973
47693,SacRT Bus,109,HAZEL EXPRESS,1824,STOCKTON BLVD & 34TH ST (WB),38.563816,-121.465904,weekday,0.000000,0.000000


In [43]:
all_ridership.to_csv(f"{GCS_FILE_PATH}ridership_all.csv", index=False)